# Model v2 Data Quality Audit

Exploratory audit for the Model v2 data flow. This notebook is read-only: it loads raw transaction and identity parquet files, merges them with `train_transaction` as the left table, and profiles numerical, categorical, and ID-like columns before further FeatureEngineeringV2 changes.

No model training, artifact writing, v1 artifact changes, or `/predict` changes are performed here.

## Setup

Paths are resolved relative to the repository root instead of hardcoded local machine paths.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ml").exists() and (candidate / "lakehouse").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TRANSACTION_PATH = PROJECT_ROOT / "lakehouse" / "raw" / "train_transaction.parquet"
IDENTITY_PATH = PROJECT_ROOT / "lakehouse" / "raw" / "train_identity.parquet"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"TRANSACTION_PATH: {TRANSACTION_PATH}")
print(f"IDENTITY_PATH: {IDENTITY_PATH}")

PROJECT_ROOT: /mnt/d/my_AI_projects/enterprise-fraud-detection-ml-system
TRANSACTION_PATH: /mnt/d/my_AI_projects/enterprise-fraud-detection-ml-system/lakehouse/raw/train_transaction.parquet
IDENTITY_PATH: /mnt/d/my_AI_projects/enterprise-fraud-detection-ml-system/lakehouse/raw/train_identity.parquet


## Load Raw Data

The transaction dataset is the source of truth for row count, `TransactionDT`, and `isFraud`. Identity data is optional enrichment keyed by `TransactionID`.

In [2]:
train_transaction = pd.read_parquet(TRANSACTION_PATH)
train_identity = pd.read_parquet(IDENTITY_PATH)

print("train_transaction shape:", train_transaction.shape)
print("train_identity shape:", train_identity.shape)

display(train_transaction.head())
display(train_identity.head())

train_transaction shape: (590540, 394)
train_identity shape: (144233, 41)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,None,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,None,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,1.0,0.0,25.0,1.0,112.0,112.0,0.0,94.0,0.0,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,50.0,1758.0,925.0,0.0,354.0,0.0,135.0,0.0,0.0,0.0,50.0,1404.0,790.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,None,NaN,NaN,NaN,None,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,NaN,100.0,NotFound,49.0,-300.0,New,NotFound,166.0,NaN,621.0,500.0,NaN,NaN,None,NaN,NaN,NaN,None,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,52.0,NaN,Found,Found,121.0,NaN,410.0,142.0,NaN,NaN,None,NaN,NaN,NaN,None,Found,Found,None,chrome 62.0,NaN,None,None,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,NaN,100.0,NotFound,52.0,NaN,New,NotFound,225.0,NaN,176.0,507.0,NaN,NaN,None,NaN,NaN,NaN,None,New,NotFound,None,chrome 62.0,NaN,None,None,F,F,T,T,desktop,None
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,NaN,-300.0,Found,Found,166.0,15.0,529.0,575.0,NaN,NaN,None,NaN,NaN,NaN,None,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


## Merge Transaction and Identity Data

Use the Model v2 strategy: keep all transaction rows and enrich available identity rows with a left join.

In [3]:
required_transaction_cols = {"TransactionID", "TransactionDT", "isFraud"}
required_identity_cols = {"TransactionID"}

missing_transaction = required_transaction_cols - set(train_transaction.columns)
missing_identity = required_identity_cols - set(train_identity.columns)

if missing_transaction:
    raise ValueError(f"Missing transaction columns: {sorted(missing_transaction)}")
if missing_identity:
    raise ValueError(f"Missing identity columns: {sorted(missing_identity)}")
if "isFraud" in train_identity.columns:
    raise ValueError("train_identity should not contain isFraud.")

merged = train_transaction.merge(train_identity, on="TransactionID", how="left")

print("merged shape:", merged.shape)
print("transaction rows preserved:", len(merged) == len(train_transaction))
print("identity columns added:", len(set(train_identity.columns) - {"TransactionID"}))
display(merged.head())

merged shape: (590540, 434)
transaction rows preserved: True
identity columns added: 40


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,None,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,None,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,1.0,0.0,25.0,1.0,112.0,112.0,0.0,94.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,None,NaN,NaN,NaN,None,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## Column Groups

Identify numerical, categorical/object, and ID-like columns for specialized audit views.

In [12]:
numeric_cols = merged.select_dtypes(include=["number", "bool"]).columns.tolist()
object_cols = merged.select_dtypes(include=["object", "category", "string"]).columns.tolist()

id_like_cols = [
    col for col in merged.columns
    if col == "TransactionID"
    or col.startswith("card")
    or col.startswith("addr")
    or col.startswith("id_")
]

print(f"numeric columns: {len(numeric_cols)}")
print(f"object/category columns: {len(object_cols)}")
print(f"ID-like columns: {len(id_like_cols)}")

display(pd.DataFrame({"id_like_columns": id_like_cols}).head(80))

numeric columns: 403
object/category columns: 31
ID-like columns: 47


,id_like_columns
0,TransactionID
1,card1
2,card2
3,card3
4,card4
5,card5
6,card6
7,addr1
8,addr2
9,id_01


## Numerical Column Audit

This summarizes dtype, missingness, conversion risk, central tendency, suspicious negatives, outliers, and constant or near-constant columns.

In [5]:
def audit_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n_rows = len(df)
    for col in df.columns:
        series = df[col]
        numeric = pd.to_numeric(series, errors="coerce")
        non_missing = series.notna().sum()
        coerced_missing = numeric.isna().sum()
        coercion_failures = max(int(coerced_missing - series.isna().sum()), 0)
        missing_count = int(series.isna().sum())
        nunique = int(series.nunique(dropna=True))
        q1 = numeric.quantile(0.25)
        q3 = numeric.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 3 * iqr if pd.notna(iqr) else np.nan
        upper = q3 + 3 * iqr if pd.notna(iqr) else np.nan
        outlier_count = int(((numeric < lower) | (numeric > upper)).sum()) if pd.notna(iqr) else 0
        top_rate = series.value_counts(dropna=True, normalize=True).iloc[0] if non_missing else np.nan
        rows.append({
            "column": col,
            "dtype": str(series.dtype),
            "missing_count": missing_count,
            "missing_rate": missing_count / n_rows if n_rows else np.nan,
            "coercion_failure_count": coercion_failures,
            "coercion_failure_rate": coercion_failures / non_missing if non_missing else 0.0,
            "min": numeric.min(),
            "max": numeric.max(),
            "mean": numeric.mean(),
            "median": numeric.median(),
            "negative_count": int((numeric < 0).sum()),
            "q01": numeric.quantile(0.01),
            "q99": numeric.quantile(0.99),
            "iqr_outlier_count": outlier_count,
            "unique_count": nunique,
            "top_value_rate": top_rate,
            "constant": nunique <= 1,
            "near_constant": bool(pd.notna(top_rate) and top_rate >= 0.99),
        })
    return pd.DataFrame(rows).sort_values(["missing_rate", "coercion_failure_rate"], ascending=False)


numeric_audit = audit_numeric_columns(merged[numeric_cols])
display(numeric_audit.head(80))

,column,dtype,missing_count,missing_rate,coercion_failure_count,coercion_failure_rate,min,max,mean,median,negative_count,q01,q99,iqr_outlier_count,unique_count,top_value_rate,constant,near_constant
399,id_24,float64,585793,0.991962,0,0.0,11.0,26.000000,12.800927,11.000000,0,11.00,21.000000,0,12,0.593427,False,False
400,id_25,float64,585408,0.991310,0,0.0,100.0,548.000000,329.608924,321.000000,0,113.00,528.000000,501,341,0.485970,False,False
386,id_07,float64,585385,0.991271,0,0.0,-46.0,61.000000,13.285354,14.000000,347,-14.46,39.000000,0,84,0.079340,False,False
387,id_08,float64,585385,0.991271,0,0.0,-100.0,0.000000,-38.600388,-34.000000,4894,-100.00,0.000000,0,94,0.096993,False,False
397,id_21,float64,585381,0.991264,0,0.0,100.0,854.000000,368.269820,252.000000,0,131.00,849.000000,0,490,0.492731,False,False
401,id_26,float64,585377,0.991257,0,0.0,100.0,216.000000,149.070308,149.000000,0,100.00,216.000000,0,95,0.159597,False,False
398,id_22,float64,585371,0.991247,0,0.0,10.0,44.000000,16.002708,14.000000,0,14.00,41.000000,433,25,0.916231,False,False
11,dist2,float64,552913,0.936284,0,0.0,0.0,11623.000000,231.855423,37.000000,0,0.00,2367.480000,2770,1751,0.151141,False,False
32,D7,float64,551623,0.934099,0,0.0,0.0,843.000000,41.638950,0.000000,0,0.00,429.000000,5941,597,0.543079,False,False
394,id_18,float64,545427,0.923607,0,0.0,10.0,29.000000,14.237337,15.000000,0,12.00,20.000000,188,18,0.565003,False,False


In [6]:
print("Columns with suspicious negative values:")
display(numeric_audit[numeric_audit["negative_count"] > 0].head(50))

print("Constant or near-constant numerical columns:")
display(numeric_audit[numeric_audit["constant"] | numeric_audit["near_constant"]].head(80))

print("High missing numerical columns:")
display(numeric_audit[numeric_audit["missing_rate"] >= 0.5].head(80))

Columns with suspicious negative values:


,column,dtype,missing_count,missing_rate,coercion_failure_count,coercion_failure_rate,min,max,mean,median,negative_count,q01,q99,iqr_outlier_count,unique_count,top_value_rate,constant,near_constant
386,id_07,float64,585385,0.991271,0,0.0,-46.0,61.0,13.285354,14.0,347,-14.46,39.0,0,84,0.079340,False,False
387,id_08,float64,585385,0.991271,0,0.0,-100.0,0.0,-38.600388,-34.0,4894,-100.00,0.0,0,94,0.096993,False,False
39,D14,float64,528353,0.894695,0,0.0,-193.0,878.0,57.724444,0.0,3,0.00,562.0,13970,802,0.727901,False,False
37,D12,float64,525823,0.890410,0,0.0,-83.0,648.0,54.037533,0.0,2,0.00,514.0,12533,635,0.660661,False,False
382,id_03,float64,524216,0.887689,0,0.0,-13.0,10.0,0.060189,0.0,187,0.00,3.0,2421,24,0.963497,False,False
383,id_04,float64,524216,0.887689,0,0.0,-28.0,0.0,-0.058938,0.0,585,0.00,0.0,585,15,0.991180,False,True
31,D6,float64,517353,0.876068,0,0.0,-83.0,873.0,69.805717,0.0,3,0.00,593.0,12665,829,0.625548,False,False
388,id_09,float64,515614,0.873123,0,0.0,-36.0,25.0,0.091023,0.0,463,0.00,3.0,4548,46,0.939300,False,False
389,id_10,float64,515614,0.873123,0,0.0,-100.0,0.0,-0.301124,0.0,2047,-10.00,0.0,2047,62,0.972680,False,False
392,id_14,float64,510496,0.864456,0,0.0,-660.0,720.0,-344.507146,-300.0,79113,-480.00,0.0,1431,25,0.551209,False,False


Constant or near-constant numerical columns:


,column,dtype,missing_count,missing_rate,coercion_failure_count,coercion_failure_rate,min,max,mean,median,negative_count,q01,q99,iqr_outlier_count,unique_count,top_value_rate,constant,near_constant
383,id_04,float64,524216,0.887689,0,0.0,-28.0,0.0,-0.058938,0.0,585,0.0,0.0,585,15,0.991180,False,True
280,V240,float64,460110,0.779134,0,0.0,0.0,7.0,1.000997,1.0,0,1.0,1.0,100,6,0.999233,False,True
281,V241,float64,460110,0.779134,0,0.0,0.0,5.0,1.000238,1.0,0,1.0,1.0,36,5,0.999724,False,True
41,V1,float64,279287,0.472935,0,0.0,0.0,1.0,0.999945,1.0,0,1.0,1.0,17,2,0.999945,False,True
81,V41,float64,168969,0.286126,0,0.0,0.0,1.0,0.999269,1.0,0,1.0,1.0,308,2,0.999269,False,True
128,V88,float64,89164,0.150987,0,0.0,0.0,1.0,0.999246,1.0,0,1.0,1.0,378,2,0.999246,False,True
129,V89,float64,89164,0.150987,0,0.0,0.0,2.0,0.000902,0.0,0,0.0,0.0,422,3,0.999158,False,True
105,V65,float64,77096,0.130552,0,0.0,0.0,1.0,0.999663,1.0,0,1.0,1.0,173,2,0.999663,False,True
108,V68,float64,77096,0.130552,0,0.0,0.0,2.0,0.000534,0.0,0,0.0,0.0,266,3,0.999482,False,True
54,V14,float64,76073,0.128819,0,0.0,0.0,1.0,0.999500,1.0,0,1.0,1.0,257,2,0.999500,False,True


High missing numerical columns:


,column,dtype,missing_count,missing_rate,coercion_failure_count,coercion_failure_rate,min,max,mean,median,negative_count,q01,q99,iqr_outlier_count,unique_count,top_value_rate,constant,near_constant
399,id_24,float64,585793,0.991962,0,0.0,11.0,26.000000,12.800927,11.000000,0,11.00,21.000000,0,12,0.593427,False,False
400,id_25,float64,585408,0.991310,0,0.0,100.0,548.000000,329.608924,321.000000,0,113.00,528.000000,501,341,0.485970,False,False
386,id_07,float64,585385,0.991271,0,0.0,-46.0,61.000000,13.285354,14.000000,347,-14.46,39.000000,0,84,0.079340,False,False
387,id_08,float64,585385,0.991271,0,0.0,-100.0,0.000000,-38.600388,-34.000000,4894,-100.00,0.000000,0,94,0.096993,False,False
397,id_21,float64,585381,0.991264,0,0.0,100.0,854.000000,368.269820,252.000000,0,131.00,849.000000,0,490,0.492731,False,False
401,id_26,float64,585377,0.991257,0,0.0,100.0,216.000000,149.070308,149.000000,0,100.00,216.000000,0,95,0.159597,False,False
398,id_22,float64,585371,0.991247,0,0.0,10.0,44.000000,16.002708,14.000000,0,14.00,41.000000,433,25,0.916231,False,False
11,dist2,float64,552913,0.936284,0,0.0,0.0,11623.000000,231.855423,37.000000,0,0.00,2367.480000,2770,1751,0.151141,False,False
32,D7,float64,551623,0.934099,0,0.0,0.0,843.000000,41.638950,0.000000,0,0.00,429.000000,5941,597,0.543079,False,False
394,id_18,float64,545427,0.923607,0,0.0,10.0,29.000000,14.237337,15.000000,0,12.00,20.000000,188,18,0.565003,False,False


## Categorical/Object Column Audit

This profiles cardinality, missingness, common values, high-cardinality fields, and mixed Python value types.

In [7]:
def audit_categorical_columns(df: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    rows = []
    n_rows = len(df)
    for col in df.columns:
        series = df[col]
        missing_count = int(series.isna().sum())
        value_counts = series.value_counts(dropna=True).head(top_n)
        type_counts = series.dropna().map(lambda x: type(x).__name__).value_counts().head(5)
        rows.append({
            "column": col,
            "dtype": str(series.dtype),
            "unique_count": int(series.nunique(dropna=True)),
            "missing_count": missing_count,
            "missing_rate": missing_count / n_rows if n_rows else np.nan,
            "top_values": value_counts.to_dict(),
            "high_cardinality": series.nunique(dropna=True) > max(100, 0.05 * n_rows),
            "observed_python_types": type_counts.to_dict(),
            "mixed_type_examples": len(type_counts) > 1,
        })
    return pd.DataFrame(rows).sort_values(["high_cardinality", "missing_rate", "unique_count"], ascending=False)


categorical_audit = audit_categorical_columns(merged[object_cols])
display(categorical_audit.head(80))

,column,dtype,unique_count,missing_count,missing_rate,top_values,high_cardinality,observed_python_types,mixed_type_examples
17,id_23,object,3,585371,0.991247,"{'IP_PROXY:TRANSPARENT': 3489, 'IP_PROXY:ANONY...",False,{'str': 5169},False
18,id_27,object,2,585371,0.991247,"{'Found': 5155, 'NotFound': 14}",False,{'str': 5169},False
23,id_33,object,260,517251,0.875895,"{'1920x1080': 16874, '1366x768': 8605, '1334x7...",False,{'str': 73289},False
21,id_30,object,75,512975,0.868654,"{'Windows 10': 21155, 'Windows 7': 13110, 'iOS...",False,{'str': 77565},False
24,id_34,object,4,512735,0.868248,"{'match_status:2': 60011, 'match_status:1': 17...",False,{'str': 77805},False
30,DeviceInfo,object,1786,471874,0.799055,"{'Windows': 47722, 'iOS Device': 19782, 'MacOS...",False,{'str': 118666},False
16,id_16,object,2,461200,0.780980,"{'Found': 66324, 'NotFound': 63016}",False,{'str': 129340},False
4,R_emaildomain,object,60,453249,0.767516,"{'gmail.com': 57147, 'hotmail.com': 27509, 'an...",False,{'str': 137291},False
22,id_31,object,130,450258,0.762451,"{'chrome 63.0': 22000, 'mobile safari 11.0': 1...",False,{'str': 140282},False
29,DeviceType,object,2,449730,0.761557,"{'desktop': 85165, 'mobile': 55645}",False,{'str': 140810},False


In [8]:
print("High-cardinality categorical columns:")
display(categorical_audit[categorical_audit["high_cardinality"]].head(80))

print("Categorical columns with mixed Python value types:")
display(categorical_audit[categorical_audit["mixed_type_examples"]].head(80))

High-cardinality categorical columns:


,column,dtype,unique_count,missing_count,missing_rate,top_values,high_cardinality,observed_python_types,mixed_type_examples


Categorical columns with mixed Python value types:


,column,dtype,unique_count,missing_count,missing_rate,top_values,high_cardinality,observed_python_types,mixed_type_examples


## ID-Like Column Audit

ID-like columns often look numeric but behave as identifiers or categorical keys. This section recommends FeatureEngineeringV2 treatment.

In [9]:
def recommend_id_treatment(col: str, unique_count: int, missing_rate: float, dtype: str) -> str:
    if col == "TransactionID":
        return "ignored"
    if col.startswith("id_"):
        return "categorical" if unique_count < 1000 else "monitored"
    if col.startswith("card") or col.startswith("addr"):
        return "categorical"
    if missing_rate >= 0.9:
        return "monitored"
    return "monitored"


def audit_id_like_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    n_rows = len(df)
    for col in columns:
        series = df[col]
        missing_count = int(series.isna().sum())
        unique_count = int(series.nunique(dropna=True))
        missing_rate = missing_count / n_rows if n_rows else np.nan
        rows.append({
            "column": col,
            "dtype": str(series.dtype),
            "unique_count": unique_count,
            "unique_rate": unique_count / n_rows if n_rows else np.nan,
            "missing_count": missing_count,
            "missing_rate": missing_rate,
            "behaves_like_unique_id": unique_count == n_rows,
            "recommended_treatment": recommend_id_treatment(col, unique_count, missing_rate, str(series.dtype)),
        })
    return pd.DataFrame(rows).sort_values(["recommended_treatment", "missing_rate", "unique_rate"], ascending=[True, False, False])


id_like_audit = audit_id_like_columns(merged, id_like_cols)
display(id_like_audit.head(120))

,column,dtype,unique_count,unique_rate,missing_count,missing_rate,behaves_like_unique_id,recommended_treatment
32,id_24,float64,12,0.000020,585793,0.991962,False,categorical
33,id_25,float64,341,0.000577,585408,0.991310,False,categorical
16,id_08,float64,94,0.000159,585385,0.991271,False,categorical
15,id_07,float64,84,0.000142,585385,0.991271,False,categorical
29,id_21,float64,490,0.000830,585381,0.991264,False,categorical
34,id_26,float64,95,0.000161,585377,0.991257,False,categorical
30,id_22,float64,25,0.000042,585371,0.991247,False,categorical
31,id_23,object,3,0.000005,585371,0.991247,False,categorical
35,id_27,object,2,0.000003,585371,0.991247,False,categorical
26,id_18,float64,18,0.000030,545427,0.923607,False,categorical


## Sample Risk Views

Small sampled views only, to keep the notebook lightweight.

In [10]:
sample_cols = [
    col for col in [
        "TransactionID", "TransactionDT", "isFraud", "TransactionAmt",
        "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo",
    ] if col in merged.columns
]

display(merged[sample_cols].sample(min(20, len(merged)), random_state=42))

,TransactionID,TransactionDT,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,P_emaildomain,R_emaildomain,DeviceType,DeviceInfo
470624,3457624,12153579,0,724.000,W,7826,481.0,150.0,mastercard,224.0,debit,387.0,87.0,aol.com,None,NaN,NaN
565820,3552820,15005886,0,108.500,W,12544,321.0,150.0,visa,226.0,debit,476.0,87.0,yahoo.com,None,NaN,NaN
284083,3271083,6970178,0,47.950,W,9400,111.0,150.0,mastercard,224.0,debit,315.0,87.0,gmail.com,None,NaN,NaN
239689,3226689,5673658,0,100.599,C,15885,545.0,185.0,visa,138.0,debit,NaN,NaN,gmail.com,gmail.com,NaN,NaN
281855,3268855,6886780,0,107.950,W,15497,490.0,150.0,visa,226.0,debit,299.0,87.0,hotmail.com,None,NaN,NaN
413908,3400908,10444930,0,280.000,W,7919,194.0,150.0,mastercard,166.0,debit,472.0,87.0,yahoo.com,None,NaN,NaN
413692,3400692,10442147,0,311.950,W,9002,453.0,150.0,visa,226.0,debit,315.0,87.0,yahoo.com,None,NaN,NaN
474182,3461182,12254683,0,330.990,W,14183,555.0,150.0,visa,226.0,credit,184.0,87.0,yahoo.com,None,NaN,NaN
370788,3357788,9228284,1,10.392,C,9633,130.0,185.0,visa,138.0,debit,NaN,NaN,icloud.com,icloud.com,desktop,Windows
28757,3015757,739585,0,335.000,W,11207,361.0,150.0,visa,226.0,debit,231.0,87.0,hotmail.com,None,NaN,NaN


## Final Summary

### Key Data Quality Risks

- Many IEEE fraud columns are sparse, especially identity/device fields after the left merge. FeatureEngineeringV2 should treat missingness as signal where appropriate.
- Card, address, and `id_*` columns may be numeric dtype but often behave like categorical identifiers rather than continuous measurements.
- High-cardinality categorical fields such as device, email, card, and identity features can create instability if encoded naively.
- Near-constant or extremely sparse columns may add noise, latency, or brittle schema behavior.
- Large ranges and outliers in amount or count-like fields should be reviewed before adding transformations or clipping.

### Recommended FeatureEngineeringV2 Validation Improvements

- Add explicit dtype policy by column group: continuous numeric, categorical identifier, ignored ID, and monitored-only fields.
- Add missing-value indicators for sparse numeric fields where missingness is predictive.
- Use `__MISSING__` and `__UNKNOWN__` handling for categorical features and high-cardinality identifiers.
- Treat `TransactionID` as an ignored join/audit key, not a model feature.
- Treat `card*`, `addr*`, and selected `id_*` fields as categorical/frequency-encoded unless there is evidence they are true measurements.
- Add checks that future-looking fields such as `uid_time_to_next` are excluded from v2.

### Recommended Monitoring Checks

- Track missing-rate drift by column group.
- Track unseen-category rates for categorical and ID-like fields.
- Track feature range and outlier drift for numerical fields.
- Track cardinality changes for card, address, email, device, and identity columns.
- Track constant or near-constant features after transformation.

### Scope Note

This notebook is exploratory only. No production code changes, model training, v1 artifact changes, `/predict` changes, or model artifact writes are performed here. Code changes should be made later only if explicitly requested.